# LifeMate — Qwen3.5-4B continuation với Unsloth

Notebook này tiếp tục huấn luyện từ adapter epoch 3 bằng **bf16 LoRA, không dùng QLoRA 4-bit**. Notebook dùng `data/combined_sft_round3.jsonl`, gồm toàn bộ combined round2 và 3.946 SFT row mới từ 1.500 utterance round3. ZIP dữ liệu round3 (giữ nguyên tên gói r2-400) chỉ chứa dữ liệu và manifest, không có model weights.

Cần tải lên Drive cả hai ZIP: gói dữ liệu round3 (giữ tên `lifemate-qwen35-pho-synthetic-error-targeted-r2-400-v1.zip`) và gói continuation v1 `lifemate-qwen35-pho-synthetic-epoch3-continue-colab-v1.zip`. Notebook lấy dữ liệu từ ZIP dữ liệu round3 (giữ nguyên tên gói r2-400) và chỉ lấy adapter epoch 3 từ ZIP continuation v1.

SFT prompt giữ nguyên English system prompt trong dữ liệu, bật thinking template, và chỉ tính loss trên action JSON gold. Cấu hình mặc định: 1 epoch, LR 1e-5, LoRA r=16/alpha=32, max length 8192, batch/GPU 1, effective batch 16. Checkpoints và adapter mới lưu trên Drive.

Trước khi chạy: chọn GPU L4 hoặc A100, bật Internet, upload cả hai ZIP vào MyDrive/lifemate, kiểm tra CFG ở cell đầu. Nếu runtime trước đó còn package cũ, dùng Runtime → Disconnect and delete runtime rồi chạy notebook mới từ đầu. Tham khảo [Unsloth Qwen3.5 fine-tuning guide](https://unsloth.ai/docs/models/qwen3.5/fine-tune).

In [ ]:
from pathlib import Path
# Các tham số đường dẫn và huấn luyện tập trung tại đây.
CFG = {
 "DRIVE_MOUNT":"/content/drive",
 "DRIVE_ROOT":"/content/drive/MyDrive/lifemate",
 "DATA_BUNDLE_NAME":"lifemate-qwen35-pho-synthetic-error-targeted-r2-400-v1.zip",
 "ADAPTER_BUNDLE_NAME":"lifemate-qwen35-pho-synthetic-epoch3-continue-colab-v1.zip",
 "DATA_PACKAGE":"lifemate-qwen35-pho-synthetic-error-targeted-r2-400-v1",
 "ADAPTER_PACKAGE":"lifemate-qwen35-pho-synthetic-epoch3-continue-colab-v1",
 "EXTRACT_ROOT":"/content/lifemate-unsloth-sft-r3-1500",
 "HF_CACHE_DIR":"/content/hf_cache",
 "RUN_NAME":"qwen35-pho-synthetic-r3-1500-unsloth-v1",
 "MODEL_ID":"Qwen/Qwen3.5-4B",
 "MODEL_REVISION":"main",
 "DATA_REL":"data/combined_sft_round3.jsonl",
 "MANIFEST_REL":"data/dataset_manifest_round3.json",
 "ADAPTER_REL":"initial_adapter",
 "DATA_SHA256":"2549ea713795d57581cb35275aa5d6f049025437d49d1e6db2de5b0acb49edaf",
 "ADAPTER_SHA256":"a15c99d66b1bc2c14846428ef0efd84865110d084fbf17f7206517ad69977048",
 "ROWS":20603, "TRAJECTORIES":1236,
 "EPOCHS":1.0, "LR":1e-5, "MAX_LENGTH":8192,
 "PER_DEVICE_BATCH":1, "EFFECTIVE_BATCH":16, "WARMUP_RATIO":0.05,
 "SAVE_STEPS":60, "SAVE_LIMIT":5, "SEED":20261114,
 "LORA_R":16, "LORA_ALPHA":32, "LORA_DROPOUT":0.05,
 "MAX_TRAIN_ROWS":None,
 "RESUME_FROM_CHECKPOINT":None,
}
CFG["DATA_BUNDLE_PATH"]=str(Path(CFG["DRIVE_ROOT"])/CFG["DATA_BUNDLE_NAME"])
CFG["ADAPTER_BUNDLE_PATH"]=str(Path(CFG["DRIVE_ROOT"])/CFG["ADAPTER_BUNDLE_NAME"])
CFG["OUTPUT_DIR"]=str(Path(CFG["DRIVE_ROOT"])/"colab-runs"/CFG["RUN_NAME"])
print(CFG)

In [ ]:
%%capture
# Cài theo setup của notebook Qwen3.5-4B chính thức của Unsloth cho Colab.
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy, _pil = f"numpy=={numpy.__version__}", f"pillow=={PIL.__version__}"
    except Exception:
        _numpy, _pil = "numpy", "pillow"
    !uv pip install -qqq "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes "xformers==0.0.32.post2" "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
from google.colab import drive
drive.mount(CFG["DRIVE_MOUNT"])
from pathlib import Path
DATA_BUNDLE=Path(CFG["DATA_BUNDLE_PATH"])
ADAPTER_BUNDLE=Path(CFG["ADAPTER_BUNDLE_PATH"])
for bundle in (DATA_BUNDLE,ADAPTER_BUNDLE):
    if not bundle.is_file(): raise FileNotFoundError(f"Upload ZIP to Drive: {bundle}")
    print(bundle.name,"size MiB:",round(bundle.stat().st_size/2**20,1))

In [ ]:
import hashlib, json, os, random, zipfile, importlib.metadata as md
import torch
def sha_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(8*1024*1024),b""): h.update(block)
    return h.hexdigest()

print({"torch":torch.__version__,"transformers":md.version("transformers"),
       "unsloth":md.version("unsloth"),"trl":md.version("trl")})
if not torch.cuda.is_available(): raise RuntimeError("Colab GPU is not enabled.")
GPU=torch.cuda.get_device_name(0)
print({"gpu":GPU,"vram_gib":round(torch.cuda.get_device_properties(0).total_memory/2**30,1),
       "bf16":torch.cuda.is_bf16_supported()})

ROOT=Path(CFG["EXTRACT_ROOT"]).resolve(); ROOT.mkdir(parents=True,exist_ok=True)
DATA_ROOT=ROOT/CFG["DATA_PACKAGE"]
with zipfile.ZipFile(DATA_BUNDLE) as z:
    if z.testzip(): raise RuntimeError("Round3 dataset ZIP is corrupt.")
    prefix=CFG["DATA_PACKAGE"]+"/"
    members=set(z.namelist())
    expected={prefix+"README.md"}
    expected|={prefix+"data/"+name for name in (
        "synthetic_error_targeted_round2_utterances_400.jsonl",
        "synthetic_error_targeted_round3_utterances_1500.jsonl",
        "synthetic_error_targeted_round2_sft.jsonl",
        "synthetic_error_targeted_round3_sft.jsonl",
        "combined_sft_round2.jsonl",
        "combined_sft_round3.jsonl",
        "dataset_manifest_round2.json",
        "dataset_manifest_round3.json",
        "error_targeted_round2_quality_audit.json",
        "error_targeted_round3_quality_audit.json",
    )}
    if members!=expected: raise RuntimeError(f"Unexpected dataset ZIP members: {sorted(members ^ expected)}")
    for item in z.infolist():
        dest=(ROOT/item.filename).resolve()
        if ROOT not in dest.parents and dest!=ROOT: raise RuntimeError("Unsafe dataset ZIP member.")
    z.extractall(ROOT)

ADAPTER=ROOT/"initial_adapter"; ADAPTER.mkdir(parents=True,exist_ok=True)
adapter_prefix=CFG["ADAPTER_PACKAGE"]+"/initial_adapter/"
adapter_members={adapter_prefix+"adapter_model.safetensors",adapter_prefix+"adapter_config.json"}
with zipfile.ZipFile(ADAPTER_BUNDLE) as z:
    available=set(z.namelist())
    if not adapter_members.issubset(available): raise RuntimeError("Adapter files are missing from continuation ZIP.")
    for member in adapter_members:
        target=ADAPTER/member.rsplit("/",1)[1]
        with z.open(member) as src, target.open("wb") as dst:
            import shutil
            shutil.copyfileobj(src,dst)

PKG=DATA_ROOT
DATA=PKG/CFG["DATA_REL"]; MANIFEST=PKG/CFG["MANIFEST_REL"]
if sha_file(DATA)!=CFG["DATA_SHA256"] or sha_file(ADAPTER/"adapter_model.safetensors")!=CFG["ADAPTER_SHA256"]:
    raise RuntimeError("Dataset or initial adapter checksum mismatch.")
M=json.loads(MANIFEST.read_text(encoding="utf-8"))
with DATA.open(encoding="utf-8") as f: ROWS=[json.loads(x) for x in f if x.strip()]
assert len(ROWS)==CFG["ROWS"]==M["combined_sft_rows"]
assert len({x["sample_id"] for x in ROWS})==CFG["TRAJECTORIES"]
assert M["validation_ids_excluded_from_training"]==309 and M["train_validation_overlap"]==0
assert M["toolenv_gold_replay"]["failures"]==0
assert all(len(x["messages"])==2 and x["messages"][0]["role"]=="system" and
           x["messages"][1]["role"]=="user" and x["target"]["action"] in {"tool_call","stop"} for x in ROWS)
PROMPTS={x["messages"][0]["content"] for x in ROWS}
assert len(PROMPTS)==1 and "You are LifeMate, a Vietnamese-language agent" in next(iter(PROMPTS))
PROMPT_SHA=hashlib.sha256(next(iter(PROMPTS)).encode()).hexdigest()
if CFG["MAX_TRAIN_ROWS"] is not None: ROWS=ROWS[:int(CFG["MAX_TRAIN_ROWS"])]
AC=json.loads((ADAPTER/"adapter_config.json").read_text())
assert AC["base_model_name_or_path"]==CFG["MODEL_ID"] and AC["r"]==CFG["LORA_R"]
print({"rows":len(ROWS),"train_trajectories":len({x['sample_id'] for x in ROWS}),
       "data_sha256":CFG["DATA_SHA256"],"adapter_sha256":CFG["ADAPTER_SHA256"],
       "prompt_sha256":PROMPT_SHA,"ToolEnv_failures":0})

## Nạp model BF16 và adapter epoch 3

Qwen3.5 là model đa phương thức nhưng tập này chỉ có text. Dùng loader FastVisionModel theo notebook Qwen3.5-4B của Unsloth; không nạp 4-bit. Adapter epoch 3 được đọc từ ZIP continuation v1 và gắn lên model nền với `is_trainable=True`.

In [ ]:
from unsloth import FastVisionModel
from peft import PeftModel

DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
os.environ["HF_HOME"]=CFG["HF_CACHE_DIR"]
os.environ["TOKENIZERS_PARALLELISM"]="false"
random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"]); torch.cuda.manual_seed_all(CFG["SEED"])

model, TOKENIZER=FastVisionModel.from_pretrained(
    model_name=CFG["MODEL_ID"],
    max_seq_length=CFG["MAX_LENGTH"],
    dtype=DTYPE,
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
    revision=CFG["MODEL_REVISION"],
)
if TOKENIZER.pad_token_id is None: TOKENIZER.pad_token=TOKENIZER.eos_token
TOKENIZER.padding_side="right"
if TOKENIZER.eos_token_id is None: raise RuntimeError("Tokenizer has no EOS token.")
model.config.use_cache=False
model=PeftModel.from_pretrained(model,str(ADAPTER),is_trainable=True)
FastVisionModel.for_training(model)
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
if trainable==0: raise RuntimeError("Initial LoRA adapter is not trainable.")
model.print_trainable_parameters()
print({"dtype":str(DTYPE),"trainable_params":trainable,
       "allocated_gib":round(torch.cuda.memory_allocated()/2**30,2)})

In [ ]:
# Giữ đúng chat template thinking của Qwen; chỉ action JSON nhận loss.
CLOSE=TOKENIZER.encode("</think>\n\n",add_special_tokens=False)
ENCODED=[]; LENS=[]; TRUNC=0
for row in ROWS:
    prompt=TOKENIZER.apply_chat_template(row["messages"],tokenize=False,
              add_generation_prompt=True,enable_thinking=True)
    pids=TOKENIZER.encode(prompt,add_special_tokens=False)
    answer=TOKENIZER.encode(json.dumps(row["target"],ensure_ascii=False,separators=(",",":")),
                            add_special_tokens=False)+[TOKENIZER.eos_token_id]
    room=CFG["MAX_LENGTH"]-len(CLOSE)-len(answer)
    if room<1: raise ValueError(f"Target over length: {row['sample_id']}/{row['step_index']}")
    if len(pids)>room:
        TRUNC+=1; head=int(room*.72); pids=pids[:head]+pids[-(room-head):]
    ids=pids+CLOSE+answer
    ENCODED.append({"input_ids":ids,"labels":[-100]*(len(pids)+len(CLOSE))+answer})
    LENS.append(len(ids))
print({"rows":len(ENCODED),"max_tokens":max(LENS),"truncated_prompts":TRUNC,
       "bf16":DTYPE==torch.bfloat16})

## Train và lưu về Drive

Dùng SFTTrainer/TRL của notebook Unsloth, nhưng giữ dataset đã token hóa và collator che loss trên prompt. Nếu gặp OOM, hạ MAX_LENGTH xuống 4096 trước; batch mỗi GPU đã là 1.

In [ ]:
from torch.utils.data import Dataset
from trl import SFTTrainer, SFTConfig

class EncodedDataset(Dataset):
    def __len__(self): return len(ENCODED)
    def __getitem__(self,i): return ENCODED[i]

class MaskedCollator:
    def __call__(self,items):
        width=((max(len(x["input_ids"]) for x in items)+7)//8)*8
        ids=[]; masks=[]; labs=[]
        for x in items:
            pad=width-len(x["input_ids"])
            ids.append(x["input_ids"]+[TOKENIZER.pad_token_id]*pad)
            masks.append([1]*len(x["input_ids"])+[0]*pad)
            labs.append(x["labels"]+[-100]*pad)
        return {"input_ids":torch.tensor(ids,dtype=torch.long),
                "attention_mask":torch.tensor(masks,dtype=torch.long),
                "labels":torch.tensor(labs,dtype=torch.long)}

OUT=Path(CFG["OUTPUT_DIR"]); CKPT=OUT/"checkpoints"
resume=CFG["RESUME_FROM_CHECKPOINT"]
if OUT.exists() and any(OUT.iterdir()) and not resume:
    raise RuntimeError(f"Output already exists; change RUN_NAME or set RESUME_FROM_CHECKPOINT: {OUT}")
OUT.mkdir(parents=True,exist_ok=True)
if resume and not Path(resume).is_dir(): raise FileNotFoundError(resume)
steps=(len(ENCODED)+CFG["EFFECTIVE_BATCH"]-1)//CFG["EFFECTIVE_BATCH"]
warm=max(1,round(CFG["WARMUP_RATIO"]*steps))
FastVisionModel.for_training(model)
trainer=SFTTrainer(
    model=model, tokenizer=TOKENIZER, train_dataset=EncodedDataset(),
    data_collator=MaskedCollator(),
    args=SFTConfig(
        output_dir=str(CKPT), num_train_epochs=CFG["EPOCHS"],
        per_device_train_batch_size=CFG["PER_DEVICE_BATCH"],
        gradient_accumulation_steps=max(1,CFG["EFFECTIVE_BATCH"]//CFG["PER_DEVICE_BATCH"]),
        learning_rate=CFG["LR"], warmup_steps=warm, lr_scheduler_type="cosine",
        weight_decay=0.01, optim="adamw_8bit",
        logging_steps=10, save_strategy="steps", save_steps=CFG["SAVE_STEPS"],
        save_total_limit=CFG["SAVE_LIMIT"], bf16=DTYPE==torch.bfloat16,
        fp16=DTYPE==torch.float16, seed=CFG["SEED"], data_seed=CFG["SEED"],
        report_to="none", remove_unused_columns=False,
        dataset_text_field="", dataset_kwargs={"skip_prepare_dataset":True},
        max_length=CFG["MAX_LENGTH"],
    ),
)
result=trainer.train(resume_from_checkpoint=resume)
FINAL=OUT/"adapter_final"
trainer.save_model(str(FINAL)); TOKENIZER.save_pretrained(FINAL)
print({"output":str(OUT),"final_adapter":str(FINAL),
       "train_runtime":result.metrics.get("train_runtime"),
       "train_loss":result.metrics.get("train_loss")})

Khi hoàn tất, adapter_final và checkpoints sẽ nằm trong thư mục run trên Drive. Báo cáo epoch 3 cũ không lưu hash chính xác của snapshot model nền Kaggle; model ID/revision Hugging Face nằm trong CFG để bạn có thể khóa lại sau khi xác minh.